### Step 1: Install Required Packages
Run this cell to make sure your Jupyter environment has the right packages installed.

In [ ]:
!pip install geopy lxml html5lib pandas requests

### Step 2: Run the Web Scraper
If this crashes, it will print the exact reason. It now features an `io.StringIO` wrapper for modern Pandas compatibility, a real-looking email User-Agent for OpenStreetMap to prevent 403s on that side, and fault-tolerance so if one team errors out, it keeps going.

In [ ]:
import pandas as pd
import time
import requests
import io
import traceback
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

try:
    print("Fetching D1 arenas list from Wikipedia...")
    url = "https://en.wikipedia.org/wiki/List_of_NCAA_Division_I_basketball_arenas"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0"}
    response = requests.get(url, headers=headers)

    # Wrap in StringIO to prevent ValueError in Pandas 2.1+
    html_io = io.StringIO(response.text)
    tables = pd.read_html(html_io)

    # Find the correct table containing the arenas. Wikipedia headers can be messy, so we use string matching.
    df = None
    for table in tables:
        # Flatten MultiIndex if it exists
        if isinstance(table.columns, pd.MultiIndex):
            table.columns = [' '.join(col).strip() for col in table.columns.values]
            
        cols = [str(c).lower() for c in table.columns]
        has_arena = any('arena' in c for c in cols)
        has_capacity = any('capacity' in c for c in cols)
        has_team = any('team' in c or 'school' in c or 'institution' in c for c in cols)
        
        if has_arena and has_team and len(table) > 100:
            df = table
            # Standardize names back to our expected ones.
            # Using ELIF because 'capacity' contains 'city' which caused a bug!
            for i, c in enumerate(cols):
                if 'arena' in c: 
                    df.columns.values[i] = 'Arena'
                elif 'capacity' in c: 
                    df.columns.values[i] = 'Capacity'
                elif 'team' in c or 'school' in c or 'institution' in c: 
                    df.columns.values[i] = 'Team'
                elif 'city' in c: 
                    df.columns.values[i] = 'City'
                elif 'state' in c: 
                    df.columns.values[i] = 'State'
            break

    if df is None:
        print("Could not find the arenas table on Wikipedia! The column names might have drastically changed.")
        print(f"Available tables had these columns: {[tuple(str(c) for c in t.columns) for t in tables[:3]]}")
    else:
        print(f"Loaded {len(df)} rows from Wikipedia (expecting around ~362).")

        output_data = []
        
        # Provide a safe unique agent. OpenStreetMap sometimes blocks default configs.
        geolocator = Nominatim(user_agent="ncaa_cbb_voronoi_map_builder_12345", timeout=10)
        geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.2)

        color_pairs = [
            ("#CC0000", "#000000"), ("#0033A0", "#FFFFFF"), ("#00274C", "#FFCB05"),
            ("#C8102E", "#F1BE48"), ("#E84A27", "#13294B"), ("#C5050C", "#FFFFFF"),
            ("#FF8200", "#FFFFFF"), ("#003087", "#FFFFFF"), ("#18453B", "#FFFFFF"),
            ("#4D1979", "#FFFFFF"), ("#FFCD00", "#000000"), ("#000000", "#BA9B37")
        ]

        print("\nGeocoding mapped arenas... (this will take ~6-7 minutes. Go grab a coffee!)")

        success = 0
        failed = 0
        
        for idx, row in df.iterrows():
            try:
                if idx > 0 and idx % 25 == 0:
                    print(f"Processed {idx} / {len(df)} teams...")
                    
                school_team = str(row.get('Team', '')).split('[')[0].strip()
                if school_team == 'nan' or school_team == '': continue
                
                # Automatically exclude women's teams since we are focusing on Men's March Madness
                if 'women' in school_team.lower():
                    continue
                    
                # Clean up any explicitly labelled "men's" tags in the dataset
                school_team = school_team.replace('(men)', '').replace('(Men)', '').replace(" Men's", "").replace(" men's", "").replace(' men', '').strip()
                
                # Note: The Wikipedia list generally does not attach mascots. 
                # Do not split by space, as that breaks "Wake Forest" and "Boston College"
                school = school_team
                mascot = "None"
                    
                arena = str(row.get('Arena', school + ' Arena')).split('[')[0].strip()
                city = str(row.get('City', '')).split('[')[0].strip()
                state = str(row.get('State', '')).split('[')[0].strip()
                
                cap_raw = str(row.get('Capacity', '5000')).replace(',', '').split('[')[0].strip()
                try:
                    capacity = int(cap_raw)
                except:
                    capacity = 5000
                    
                location = None
                query = f"{arena}, {city}, {state}"
                try:
                    location = geocode(query)
                    if location is None:
                        location = geocode(f"{school} University, {city}, {state}")
                    if location is None:
                        location = geocode(f"{city}, {state}")
                except Exception as geo_e:
                    print(f"Geocoding WARNING for {school}: {geo_e}")

                if location:
                    lat = location.latitude
                    lon = location.longitude
                    success += 1
                else:
                    lat, lon = 39.8283, -98.5795
                    failed += 1
                    
                c1, c2 = color_pairs[idx % len(color_pairs)]
                
                output_data.append({
                    'School': school,
                    'Mascot': mascot,
                    'Arena': arena,
                    'Hex': c1,
                    'Hex2': c2,
                    'Lat': round(lat, 4),
                    'Lon': round(lon, 4),
                    'Furthest_Round': 362,
                    'Capacity': capacity,
                    'Avg_Attendance': capacity
                })
            except Exception as row_error:
                # Silent skip, bad row
                pass

        print(f"\nGeocoding complete! Success: {success}, Fallbacks used: {failed}")

        final_df = pd.DataFrame(output_data)
        final_df.to_csv("ncaa_d1_teams_initial.csv", index=False)
        print(f"✅ Saved all {len(final_df)} NCAA teams to 'ncaa_d1_teams_initial.csv'!")

except Exception as main_e:
    print("\n--- EXCEPTION CAUGHT ---")
    traceback.print_exc()
